# Cuaderno U2-01. Software para modelación y simulación

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2
**Unidad 2.** Herramientas computacionales para modelación y simulación
**Subtema del plan.** 2.1 Introducción al software especializado para modelación y simulación
**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad2/U2_01_software_para_modelacion.ipynb)

La insignia anterior queda con la dirección del repositorio pendiente. El
docente reemplaza `msc-unisucre/msc2026-material` por la ruta real antes de publicar.

Este cuaderno recorre la Sección 2.1 del libro. No presenta las
bibliotecas una por una, sino que las pone a trabajar en la tarea que a cada
una le corresponde, según la Tabla 2.1 del libro, y monta el proyecto
reproducible de la Figura 2.3 con la política de versionado de la Tabla 2.3.
La teoría no se repite, se ejecuta.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante debe ser capaz de lo siguiente.

1. Asignar a cada tarea de un proyecto de modelación el estrato del ecosistema que la resuelve, siguiendo la Tabla 2.1 del libro.
2. Registrar el entorno de ejecución y la semilla junto con los resultados, reproduciendo el Listado 2.1 del libro.
3. Montar la estructura de carpetas de la Tabla 2.3 con rutas relativas y justificar qué entra al repositorio y qué no.
4. Demostrar con la semilla 20262 que un resultado aleatorio se reproduce, y explicar por qué eso no lo hace correcto.
5. Resolver un mismo modelo empleando cada estrato en su tarea y verificar el resultado contra la cifra que publica el libro.

## Puesta a punto

La primera celda instala lo que falte y la segunda fija la semilla del curso,
la paleta del libro y la función que compara cada resultado con el valor
publicado. Ningún resultado de este cuaderno depende de una ejecución
concreta.

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        *faltantes], check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
# Configuración común a todos los cuadernos del curso.
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
          "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110,
                     "font.size": 9, "axes.grid": True,
                     "grid.linewidth": 0.4, "grid.alpha": 0.5,
                     "axes.prop_cycle": plt.cycler(color=list(PALETA.values()))})


def contra_libro(nombre: str, calculado: float, publicado: float,
                 unidad: str = "", tol: float = 1e-3, relativa: bool = True,
                 exigir: bool = True, nota: str = "") -> None:
    """Compara un resultado del cuaderno con el valor que publica el libro.

    Detiene la ejecución si la diferencia excede la tolerancia y `exigir` es
    verdadero. Las magnitudes que dependen de la máquina, como los tiempos de
    ejecución, se informan con `exigir=False` y una nota que lo advierte.
    """
    error = abs(calculado - publicado)
    if relativa and publicado != 0.0:
        error = error / abs(publicado)
    ok = error <= tol
    print(f"{nombre:<44s} cuaderno {calculado:>13.6g}  "
          f"libro {publicado:>13.6g} {unidad:<9s} "
          f"{'coincide' if ok else 'DIFIERE '}{nota}")
    if exigir and not ok:
        raise AssertionError(
            f"{nombre}, el cuaderno da {calculado!r} y el libro publica "
            f"{publicado!r}, con error {error:.3e}")


print("NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| pandas", pd.__version__, "| SymPy", sp.__version__)

## 1. Los estratos del ecosistema

La Figura 2.2 del libro dispone el ecosistema científico de Python en estratos
independientes, cada uno con una responsabilidad única, y advierte qué no debe
pedírsele a cada uno. La Tabla 2.1 traduce esa figura a una correspondencia
entre la tarea del proyecto y el estrato responsable. La celda siguiente
reproduce esa tabla y declara la versión instalada de cada estrato, que es la
información mínima que exige la Definición 2.2 del libro.

In [ ]:
import matplotlib
import platform
import sys

ESTRATOS = pd.DataFrame(
    [("Ingresar y depurar datos", "pandas", "DataFrame", "cálculo fila a fila"),
     ("Representar campos", "NumPy", "ndarray", "bucles explícitos"),
     ("Resolver el modelo", "SciPy", "rutina especializada",
      "tolerancia por omisión"),
     ("Derivar y linealizar", "SymPy", "expresión exacta",
      "explosión simbólica"),
     ("Diagnosticar resultados", "Matplotlib", "Figure, Axes",
      "figura sin unidades"),
     ("Documentar el análisis", "Jupyter", "cuaderno",
      "ejecución desordenada"),
     ("Versionar el trabajo", "Git", "repositorio", "versionar salidas")],
    columns=["Tarea del proyecto", "Estrato", "Objeto central",
             "Riesgo de mal uso"])

VERSIONES = {"python": sys.version.split()[0], "numpy": np.__version__,
             "scipy": scipy.__version__, "pandas": pd.__version__,
             "matplotlib": matplotlib.__version__, "sympy": sp.__version__}

print(ESTRATOS.to_string(index=False))
print()
for clave, valor in VERSIONES.items():
    print(f"  {clave:<12s} {valor}")

## 2. Un mismo modelo, cada estrato en su tarea

El Listado 2.9 del libro escribe un quimiostato como una estructura inmutable
de parámetros y una función pura que devuelve la derivada. Sobre ese modelo se
reparten las tareas. SymPy obtiene el estado estacionario de forma exacta,
NumPy y SciPy lo integran, pandas organiza la salida y Matplotlib la
diagnostica. El libro publica en la Sección 2.4 el estado estacionario con
sustrato residual de 15.0 mg/L y biomasa de 329.7 mg/L, de modo que ese par de
cifras sirve de verificación.

In [ ]:
from dataclasses import dataclass

from scipy.integrate import solve_ivp


@dataclass(frozen=True)
class Quimiostato:
    """Parámetros del quimiostato del Listado 2.9 del libro."""

    mu_max: float = 0.40        # 1/d
    K_s: float = 25.0           # mg/L
    Y: float = 0.42             # mg biomasa por mg sustrato
    D: float = 0.15             # 1/d
    S_in: float = 800.0         # mg/L


def derivada(t: float, y: np.ndarray, p: Quimiostato) -> np.ndarray:
    """Devuelve dy/dt del modelo, con y = (X, S) en mg/L."""
    X, S = y
    mu = p.mu_max * S / (p.K_s + S)
    return np.array([(mu - p.D) * X, p.D * (p.S_in - S) - mu * X / p.Y])


par = Quimiostato()
print(par)
print("derivada en (X, S) = (300, 20) mg/L por día:",
      np.round(derivada(0.0, np.array([300.0, 20.0]), par), 4))

In [ ]:
# SymPy resuelve el estado estacionario de forma exacta.
Xs, Ss = sp.symbols("X S", positive=True)
mu_s = par.mu_max * Ss / (par.K_s + Ss)
ecuaciones = [sp.Eq((mu_s - par.D) * Xs, 0),
              sp.Eq(par.D * (par.S_in - Ss) - mu_s * Xs / par.Y, 0)]
estacionario = sp.solve(ecuaciones, [Xs, Ss], dict=True)[0]
X_est = float(estacionario[Xs])
S_est = float(estacionario[Ss])

contra_libro("sustrato residual S*", S_est, 15.0, "mg/L")
contra_libro("biomasa X*", X_est, 329.7, "mg/L")

In [ ]:
# SciPy integra el transitorio y pandas organiza la salida.
solucion = solve_ivp(derivada, (0.0, 120.0), np.array([50.0, 700.0]),
                     args=(par,), rtol=1e-9, atol=1e-11, dense_output=True)
malla = np.linspace(0.0, 120.0, 481)
trayectoria = pd.DataFrame(solucion.sol(malla).T, columns=["X", "S"],
                           index=pd.Index(malla, name="t_d"))

final = trayectoria.iloc[-1]
print(trayectoria.iloc[::120].round(3).to_string())
print()
contra_libro("S a los 120 d", float(final["S"]), 15.0, "mg/L", tol=2e-3)
contra_libro("X a los 120 d", float(final["X"]), 329.7, "mg/L", tol=2e-3)

In [ ]:
# Matplotlib diagnostica. La referencia física entra como línea horizontal.
fig, ax = plt.subplots(figsize=(13.5 / 2.54, 6.4 / 2.54), layout="constrained")
ax.plot(trayectoria.index, trayectoria["X"], color=PALETA["azul"], lw=1.4,
        label="biomasa X")
ax.plot(trayectoria.index, trayectoria["S"], color=PALETA["rojo"], lw=1.4,
        ls="--", label="sustrato S")
ax.axhline(X_est, color=PALETA["gris"], lw=0.8, ls=":")
ax.axhline(S_est, color=PALETA["gris"], lw=0.8, ls=":")
ax.set_xlabel("Tiempo (d)")
ax.set_ylabel("Concentración (mg/L)")
ax.set_xlim(0.0, 120.0)
ax.set_ylim(0.0, 750.0)
ax.legend(loc="center right")
plt.show()

## 3. Reproducibilidad computacional

La Definición 2.1 del libro exige que un tercero obtenga los mismos números
con los datos, el código y la descripción del entorno. La definición no exige
que el modelo sea correcto, y esa particularidad es la que permite descubrir
el error. La celda siguiente lo muestra de las dos maneras. Primero comprueba
que la semilla del curso reproduce el mismo muestreo, y después estima una
media con una fórmula equivocada, que resulta reproducible hasta el último
dígito y falsa de todos modos.

In [ ]:
def muestreo(semilla: int, n: int = 5) -> np.ndarray:
    """Caudales simulados en L/s con el generador declarado."""
    return np.random.default_rng(semilla).normal(45.0, 2.0, n)


a = muestreo(SEMILLA)
b = muestreo(SEMILLA)
c = muestreo(SEMILLA + 1)

print("misma semilla   ", np.round(a, 4))
print("misma semilla   ", np.round(b, 4))
print("otra semilla    ", np.round(c, 4))
assert np.array_equal(a, b), "la semilla no está fijando el muestreo"
assert not np.array_equal(a, c), "dos semillas distintas no deben coincidir"

# Reproducible y equivocado. La varianza poblacional divide entre n y no
# entre n - 1, de modo que subestima la varianza muestral siempre igual.
muestra = np.random.default_rng(SEMILLA).normal(45.0, 2.0, 30)
s_mal = float(np.sqrt(((muestra - muestra.mean()) ** 2).sum() / muestra.size))
s_bien = float(muestra.std(ddof=1))
print(f"\nestimador equivocado {s_mal:.6f} L/s, correcto {s_bien:.6f} L/s")
print("el equivocado se reproduce con exactitud en cualquier máquina, "
      "y sigue estando mal")

## 4. Entorno declarativo y registro de la semilla

El Listado 2.1 del libro deja constancia del entorno y de la semilla dentro de
los propios resultados. El archivo pesa unos pocos cientos de bytes y responde
por adelantado la pregunta de quien intente repetir el trabajo. La celda
siguiente lo reproduce sobre una carpeta de trabajo creada con rutas
relativas.

In [ ]:
import json

PROYECTO = Path("salida") / "proyecto_demo"
(PROYECTO / "resultados").mkdir(parents=True, exist_ok=True)

entorno = {"python": sys.version.split()[0],
           "plataforma": platform.platform(),
           "numpy": np.__version__, "scipy": scipy.__version__,
           "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
           "sympy": sp.__version__, "semilla": SEMILLA}

with (PROYECTO / "resultados" / "entorno.json").open("w",
                                                     encoding="utf-8") as f:
    json.dump(entorno, f, indent=2, ensure_ascii=False)

leido = json.loads((PROYECTO / "resultados" / "entorno.json").read_text(
    encoding="utf-8"))
assert leido == entorno, "el archivo de entorno no coincide con lo escrito"
print(json.dumps(leido, indent=2, ensure_ascii=False))
print("\ntamaño del archivo:",
      (PROYECTO / "resultados" / "entorno.json").stat().st_size, "bytes")

## 5. Estructura del proyecto reproducible

La Figura 2.3 y la Tabla 2.3 del libro fijan las carpetas, su contenido y la
regla de versionado. Se versiona todo lo que un ser humano escribió y no puede
reconstruirse, y no se versiona nada que una orden pueda regenerar. La celda
siguiente construye ese árbol y escribe el archivo de exclusiones coherente
con la tabla.

In [ ]:
TABLA_PROYECTO = pd.DataFrame(
    [("datos/crudo/", "registros tal como llegaron", True, "nunca"),
     ("datos/procesado/", "tablas depuradas", False, "por el guion"),
     ("src/", "funciones del modelo", True, "sí"),
     ("cuadernos/", "narrativa del análisis", True, "sí"),
     ("resultados/", "tablas y valores calculados", False, "por el guion"),
     ("figuras/", "gráficas exportadas", False, "por el guion"),
     ("entorno.yml", "versiones del entorno", True, "sí"),
     ("README.md", "propósito y orden de ejecución", True, "sí")],
    columns=["Carpeta o archivo", "Contenido", "Versionado", "Modificable"])

for entrada in TABLA_PROYECTO["Carpeta o archivo"]:
    destino = PROYECTO / entrada
    if entrada.endswith("/"):
        destino.mkdir(parents=True, exist_ok=True)
    else:
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.touch()

excluidos = [e for e in TABLA_PROYECTO.loc[~TABLA_PROYECTO["Versionado"],
                                           "Carpeta o archivo"]]
(PROYECTO / ".gitignore").write_text("\n".join(excluidos) + "\n",
                                     encoding="utf-8")

print(TABLA_PROYECTO.to_string(index=False))
print("\ncontenido de .gitignore")
print((PROYECTO / ".gitignore").read_text(encoding="utf-8"))

In [ ]:
# El entorno declarativo de la Definición 2.2 y el archivo de propósito.
(PROYECTO / "entorno.yml").write_text(
    "name: msc-2026-2\n"
    "channels:\n  - conda-forge\n"
    "dependencies:\n"
    f"  - python={sys.version.split()[0]}\n"
    f"  - numpy={np.__version__}\n"
    f"  - scipy={scipy.__version__}\n"
    f"  - pandas={pd.__version__}\n"
    f"  - matplotlib={matplotlib.__version__}\n"
    f"  - sympy={sp.__version__}\n", encoding="utf-8")

(PROYECTO / "README.md").write_text(
    "# Proyecto de demostración\n\n"
    "Estación de bombeo del acueducto municipal.\n\n"
    "## Orden de ejecución\n\n"
    "1. `src/preparar.py` produce `datos/procesado/` desde `datos/crudo/`.\n"
    "2. `cuadernos/` documenta el análisis.\n"
    "3. `src/figuras.py` regenera `figuras/` y `resultados/`.\n\n"
    f"Semilla del curso {SEMILLA}. Entorno declarado en `entorno.yml`.\n",
    encoding="utf-8")

arbol = sorted(p.relative_to(PROYECTO).as_posix()
               for p in PROYECTO.rglob("*"))
print("\n".join(arbol))

## 6. La trampa del cuaderno

El libro advierte que las celdas pueden ejecutarse en cualquier orden y que el
estado de la sesión sobrevive a la edición del código, de modo que un cuaderno
puede mostrar resultados que ninguna ejecución ordenada reproduce. La celda
siguiente simula ese desorden dentro de una sola ejecución, para que quede a
la vista sin necesidad de romper el cuaderno. La disciplina mínima consiste en
reiniciar el núcleo y ejecutar todo de arriba abajo antes de dar por buena
cualquier salida.

In [ ]:
def sesion(orden: list[str]) -> float:
    """Simula la ejecución de tres celdas en el orden indicado."""
    estado = {"acumulado": 0.0}

    def celda_a() -> None:
        estado["acumulado"] = 10.0

    def celda_b() -> None:
        estado["acumulado"] = estado["acumulado"] * 2.0

    def celda_c() -> None:
        estado["acumulado"] = estado["acumulado"] + 5.0

    celdas = {"a": celda_a, "b": celda_b, "c": celda_c}
    for nombre in orden:
        celdas[nombre]()
    return estado["acumulado"]


print("orden a, b, c        ->", sesion(["a", "b", "c"]))
print("orden a, c, b        ->", sesion(["a", "c", "b"]))
print("orden a, b, c, b     ->", sesion(["a", "b", "c", "b"]))
print("\nlas tres salidas son plausibles y solo una corresponde a una "
      "ejecución ordenada")

### Acceso a los datos

Los archivos viven en `03_cuadernos/datos/`. La función `ruta_datos` intenta
primero la ruta relativa del repositorio y, si el archivo no está, lo regenera
con la semilla del curso. Así el cuaderno corre igual en Colab, donde no
existe la carpeta, y en una instalación local. Nunca se usan rutas absolutas
del computador del docente.

In [ ]:
# Acceso a los datos. Se intenta la ruta relativa del repositorio y, si el
# archivo no existe, se regenera con la semilla del curso.
EXTRACTO_CAUDAL = [
    ("2026-03-04 08:00:00", "41,8", "3,42"),
    ("2026-03-04 08:05:00", "42,3", "3,40"),
    ("2026-03-04 08:10:00", "43,1", "3,38"),
    ("2026-03-04 08:15:00", "-9,9", "3,41"),
    ("2026-03-04 08:20:00", "44,0", "3,36"),
    ("2026-03-04 08:25:00", "", "3,35"),
    ("2026-03-04 08:30:00", "43,6", "3,37"),
    ("2026-03-04 08:35:00", "186,0", "3,33"),
    ("2026-03-04 08:45:00", "44,5", "3,31"),
    ("2026-03-04 08:40:00", "43,9", "3,34"),
    ("2026-03-04 08:50:00", "44,2", "3,30"),
    ("2026-03-04 08:50:00", "43,9", "3,30"),
]


def _gen_caudal_bombeo(destino: Path) -> None:
    """Extracto sucio de doce registros del Ejemplo 2.1 del libro."""
    lineas = ["fecha_hora;caudal_l_s;presion_bar"]
    lineas += [";".join(fila) for fila in EXTRACTO_CAUDAL]
    destino.write_text("\n".join(lineas) + "\n", encoding="utf-8")


GENERADORES = {
    "caudal_bombeo.csv": _gen_caudal_bombeo,
}

CANDIDATAS = [Path("datos"), Path("..") / "datos",
              Path("03_cuadernos") / "datos", Path("..") / ".." / "datos"]


def ruta_datos(nombre: str) -> Path:
    """Devuelve la ruta del archivo de datos, generándolo si hace falta."""
    for base in CANDIDATAS:
        candidata = base / nombre
        if candidata.is_file():
            return candidata
    generada = Path("salida") / "datos_generados"
    generada.mkdir(parents=True, exist_ok=True)
    destino = generada / nombre
    if not destino.is_file():
        GENERADORES[nombre](destino)
    return destino

print("datos disponibles en", ruta_datos("caudal_bombeo.csv").parent)

In [ ]:
ruta = ruta_datos("caudal_bombeo.csv")
print("ruta relativa usada:", ruta.as_posix())
print(ruta.read_text(encoding="utf-8"))

## 7. Ejercicios guiados

Cinco celdas incompletas. Cada una lleva una marca de completar con la
descripción precisa de lo que falta y una celda de verificación
inmediatamente después. El cuaderno sigue ejecutándose aunque no se
completen, porque la verificación queda desactivada con la bandera `REVISAR`.
Al completar una celda debe cambiarse esa bandera a `True`.

### Ejercicio 1. Registro del entorno

Construya el diccionario del entorno tal como lo hace el Listado 2.1 del
libro, con las claves `python`, `plataforma`, `numpy`, `scipy`, `pandas`,
`matplotlib`, `sympy` y `semilla`.

In [ ]:
# COMPLETE: arme el diccionario `entorno_ej` con las ocho claves indicadas.
# El valor de `semilla` debe ser la semilla del curso.
REVISAR_1 = False
entorno_ej = {"python": "desconocido"}     # <- reemplace por el diccionario

In [ ]:
CLAVES = {"python", "plataforma", "numpy", "scipy", "pandas", "matplotlib",
          "sympy", "semilla"}
if REVISAR_1:
    assert set(entorno_ej) == CLAVES, f"faltan o sobran claves: {set(entorno_ej) ^ CLAVES}"
    assert entorno_ej["semilla"] == SEMILLA, "la semilla no es la del curso"
    assert entorno_ej["numpy"] == np.__version__, "la versión de NumPy no coincide"
    print("ejercicio 1 correcto, el entorno queda declarado")
else:
    print("ejercicio 1 pendiente, complete la celda y ponga REVISAR_1 = True")

### Ejercicio 2. Estrato responsable de cada tarea

Escriba la función que devuelve el estrato que resuelve una tarea, según la
Tabla 2.1 del libro.

In [ ]:
# COMPLETE: devuelva el estrato de la Tabla 2.1 que corresponde a la tarea.
# Las tareas admitidas son las siete de la columna izquierda de la tabla.
REVISAR_2 = False


def estrato_de(tarea: str) -> str:
    """Estrato responsable de una tarea del proyecto."""
    return "Jupyter"      # <- reemplace por la consulta a la tabla

In [ ]:
if REVISAR_2:
    esperado = dict(zip(ESTRATOS["Tarea del proyecto"], ESTRATOS["Estrato"]))
    for tarea, estrato in esperado.items():
        obtenido = estrato_de(tarea)
        assert obtenido == estrato, f"{tarea} -> {obtenido}, se esperaba {estrato}"
    try:
        estrato_de("Comprar una licencia")
    except ValueError:
        print("ejercicio 2 correcto, incluida la tarea no contemplada")
    else:
        raise AssertionError("una tarea ajena a la tabla debe levantar ValueError")
else:
    print("ejercicio 2 pendiente, complete la celda y ponga REVISAR_2 = True")

### Ejercicio 3. Política de versionado

Escriba la función que decide si una ruta del proyecto entra al repositorio,
según la Tabla 2.3 del libro. La regla es que se versiona lo que una persona
escribió y no puede reconstruirse.

In [ ]:
# COMPLETE: devuelva True si la ruta debe versionarse y False si una orden
# puede regenerarla. Use la tabla TABLA_PROYECTO como fuente.
REVISAR_3 = False


def se_versiona(entrada: str) -> bool:
    """Decide si una entrada del proyecto entra al repositorio."""
    return True           # <- reemplace por la consulta a la tabla

In [ ]:
if REVISAR_3:
    for entrada, esperado in zip(TABLA_PROYECTO["Carpeta o archivo"],
                                 TABLA_PROYECTO["Versionado"]):
        assert se_versiona(entrada) == esperado, f"decisión errada en {entrada}"
    assert not se_versiona("figuras/"), "las figuras se regeneran, no se versionan"
    assert se_versiona("datos/crudo/"), "el dato crudo no puede reconstruirse"
    print("ejercicio 3 correcto, la política coincide con la Tabla 2.3")
else:
    print("ejercicio 3 pendiente, complete la celda y ponga REVISAR_3 = True")

### Ejercicio 4. Comprobación de reproducibilidad

Escriba la función que ejecuta dos veces un cálculo con generación aleatoria y
devuelve si las dos corridas coinciden dígito a dígito.

In [ ]:
# COMPLETE: devuelva True solo si dos llamadas independientes con la misma
# semilla producen exactamente el mismo arreglo. No use tolerancia.
REVISAR_4 = False


def es_reproducible(funcion, semilla: int) -> bool:
    """Comprueba que dos corridas con la misma semilla den lo mismo."""
    return False          # <- reemplace por la comprobación

In [ ]:
if REVISAR_4:
    assert es_reproducible(muestreo, SEMILLA), "el muestreo declarado sí es reproducible"

    contador = {"n": 0}

    def con_estado(semilla: int) -> np.ndarray:
        """Depende de un estado externo, de modo que no es reproducible."""
        contador["n"] += 1
        return np.random.default_rng(semilla + contador["n"]).normal(0, 1, 3)

    assert not es_reproducible(con_estado, SEMILLA), \
        "una función con estado externo no debe pasar la comprobación"
    print("ejercicio 4 correcto, la comprobación distingue los dos casos")
else:
    print("ejercicio 4 pendiente, complete la celda y ponga REVISAR_4 = True")

### Ejercicio 5. Estado estacionario con lavado del reactor

El quimiostato se lava cuando la tasa de dilución supera la máxima de
crecimiento, y entonces el único estado estacionario es el de biomasa nula con
sustrato igual al de alimentación. Calcule el estado estacionario para una
tasa de dilución de 0.55 por día.

In [ ]:
# COMPLETE: calcule el par (X*, S*) del quimiostato con D = 0.55 1/d.
# Recuerde que si D supera mu_max no existe estado con biomasa positiva.
REVISAR_5 = False
par_lavado = Quimiostato(D=0.55)
X_lavado, S_lavado = 0.0, 0.0     # <- reemplace por el cálculo

In [ ]:
if REVISAR_5:
    residuo = derivada(0.0, np.array([max(X_lavado, 1e-12), S_lavado]),
                       par_lavado)
    assert abs(S_lavado - par_lavado.S_in) < 1e-9, \
        "con lavado el sustrato de salida iguala al de alimentación"
    assert abs(X_lavado) < 1e-9, "con lavado la biomasa se anula"
    assert np.abs(residuo).max() < 1e-6, \
        f"la derivada no se anula, vale {residuo}"
    print(f"ejercicio 5 correcto, X* = {X_lavado:.1f} mg/L y "
          f"S* = {S_lavado:.1f} mg/L, derivada nula")
else:
    print("ejercicio 5 pendiente, complete la celda y ponga REVISAR_5 = True")

## 8. Problemas del capítulo

Se abordan tres problemas de la Sección 2.7 del libro. El 2-1 y el 2-2 son de
apropiación conceptual y se responden con evidencia calculada aquí mismo. El
2-26 es de diseño y queda planteado como andamiaje.

### Problema 2-1

Explique por qué un análisis reproducible puede ser erróneo, y por qué la
reproducibilidad sigue siendo condición necesaria del trabajo profesional.

La Sección 3 de este cuaderno construyó la evidencia. El estimador que divide
entre `n` en lugar de `n - 1` produce siempre el mismo número en cualquier
máquina y subestima la desviación estándar de manera sistemática. La celda
siguiente cuantifica ese sesgo sobre mil réplicas, de modo que la respuesta al
problema quede sostenida por una cifra y no por una opinión.

In [ ]:
replicas = 1000
sesgos = np.empty(replicas)
generador = np.random.default_rng(SEMILLA)
for i in range(replicas):
    m = generador.normal(45.0, 2.0, 30)
    sesgos[i] = np.sqrt(((m - m.mean()) ** 2).mean()) - m.std(ddof=1)

print(f"sesgo medio del estimador equivocado {sesgos.mean():+.4f} L/s")
print(f"réplicas con sesgo negativo {int((sesgos < 0).sum())} de {replicas}")
print("\nel resultado es reproducible dígito a dígito y sesgado siempre "
      "hacia abajo, de modo que la reproducibilidad es lo que permite "
      "detectar el defecto en lugar de esconderlo")

### Problema 2-2

Un colega dice que su proyecto es reproducible porque guardó las figuras en el
repositorio. Tres razones por las cuales eso es falso.

1. Una figura versionada no dice con qué código ni con qué datos se produjo,
   de modo que no cumple la Definición 2.1, que exige el camino completo desde
   el dato hasta la conclusión.
2. Los archivos binarios de figura no admiten comparación línea a línea, de
   suerte que el repositorio crece y los conflictos se vuelven irresolubles,
   tal como advierte la Sección 2.1 del libro.
3. Nada garantiza que la figura guardada corresponda a la última versión del
   código, y esa duda permanente es exactamente lo que la regla de no
   versionar lo regenerable viene a eliminar. La celda siguiente lo muestra
   con la marca de tiempo de dos ejecuciones del mismo guion.

In [ ]:
figura_guardada = PROYECTO / "figuras" / "quimiostato.png"
figura_guardada.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figura_guardada)

codigo = PROYECTO / "src" / "figuras.py"
codigo.parent.mkdir(parents=True, exist_ok=True)
codigo.write_text("# guion que regenera la figura\n", encoding="utf-8")

print(f"figura   {figura_guardada.stat().st_size:>8d} bytes")
print(f"código   {codigo.stat().st_size:>8d} bytes")
print("\nel repositorio guarda el archivo pesado y sigue sin poder responder "
      "qué versión del código lo produjo, mientras que versionar el guion, "
      "mucho más liviano, responde esa pregunta y regenera la figura")

### Problema 2-26

Diseñe un proyecto reproducible para el monitoreo de calidad de agua de un
río, con carpetas, entorno declarativo, descripción y política de versionado
justificados.

El andamiaje está montado en la Sección 5 de este cuaderno. Para resolver el
problema el estudiante debe adaptar `TABLA_PROYECTO` al caso del río, es
decir, separar los registros de las estaciones de aforo de los de laboratorio,
declarar las unidades de cada variable, fijar el paso nominal de muestreo y
justificar cada decisión de versionado con la regla de la Sección 2.1 del
libro. La entrega es la carpeta con su archivo de propósito, su entorno
declarativo y su archivo de exclusiones, más media página que explique por qué
cada carpeta está donde está.

## Cierre

### Lista de comprobación

Al cerrar el cuaderno el estudiante debe poder hacer lo siguiente sin
consultar la solución.

- Nombrar el estrato responsable de cada una de las siete tareas de la Tabla 2.1 y el riesgo de mal uso asociado.
- Escribir el registro del entorno del Listado 2.1 y explicar qué pregunta responde ese archivo.
- Montar el árbol de la Tabla 2.3 con rutas relativas y redactar el archivo de exclusiones coherente con él.
- Demostrar que un cálculo con generación aleatoria se reproduce con la semilla 20262.
- Explicar con un caso propio por qué reproducible no significa correcto.

### Qué revisar en el libro si algo no salió

- Si la asignación de estratos no salió, la Tabla 2.1 y la Figura 2.2 de la Sección 2.1.
- Si el registro del entorno no salió, el Listado 2.1 y la Definición 2.2.
- Si la política de versionado no salió, la Figura 2.3 y la Tabla 2.3.
- Si la reproducibilidad se confundió con la corrección, la Definición 2.1 y el párrafo que la sigue.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente automático de programación.
Todo fragmento se sometió al protocolo del Algoritmo 2.3 del libro y cada
resultado numérico se comprueba contra la cifra publicada mediante la función
`contra_libro`. La regla de la asignatura es que el ingeniero responde por el
resultado que firma, con independencia de quién haya tecleado las líneas.